[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/20_Dive_Computers.ipynb)

# DiveLab

## Notebook 20 — Dive Computers as Measurement-and-Model Systems

**Guiding question:** What does a dive computer actually measure, what does it estimate, and what does it predict?

A dive computer is not just a screen.

Conceptually, it is a pipeline:

$$
\boxed{
\text{pressure sensor}
\rightarrow
\text{depth estimate}
\rightarrow
\text{time-depth history}
\rightarrow
\text{models}
\rightarrow
\text{displayed information}
}
$$

This notebook connects sensors, estimation, numerical integration, gas consumption, and model-based inference.

## Learning objectives

By the end of this notebook, you will be able to:

- distinguish measured quantities from estimated quantities;
- convert pressure measurements into depth;
- understand sampling and time-depth histories;
- compute ascent rate from depth history;
- understand why differentiation amplifies noise;
- use filtering for smoother estimates;
- integrate depth-time exposure;
- distinguish direct measurements from model outputs;
- understand how gas and decompression estimates depend on models;
- interpret a dive computer as an observer-like system.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. What does the computer measure directly?

At its core, a dive computer usually has a pressure sensor.

Conceptually, the sensor produces:

$$
P_m(t)
$$

where the subscript $m$ means measured.

The computer then converts pressure to depth.

So depth itself is already an **estimated quantity derived from pressure**.

# 2. Hydrostatic conversion

Use:

$$
P(z)
=
P_0+\rho gz.
$$

Therefore:

$$
\boxed{
z
=
\frac{P-P_0}{\rho g}
}
$$

In [ ]:
rho = 1025.0
g = 9.80665
P0 = 101325.0

def pressure_from_depth(depth_m):
    return P0 + rho*g*depth_m

def depth_from_pressure(pressure_pa):
    return (pressure_pa - P0)/(rho*g)

# 3. Simulate a dive profile

We create a simple profile:

- descent;
- bottom segment;
- ascent;
- shallow hold.

In [ ]:
def true_depth_profile(t_s):
    if t_s < 120:
        return 20/120 * t_s

    if t_s < 900:
        return 20.0

    if t_s < 1100:
        return 20 - 15*(t_s-900)/200

    if t_s < 1280:
        return 5.0

    if t_s < 1340:
        return 5 - 5*(t_s-1280)/60

    return 0.0

In [ ]:
t = np.arange(0, 1400, 1.0)
z_true = np.array([true_depth_profile(tt) for tt in t])

plt.plot(t/60, z_true)
plt.gca().invert_yaxis()
plt.xlabel("Time [min]")
plt.ylabel("Depth [m]")
plt.title("True dive profile")
plt.grid(True)
plt.show()

# 4. Pressure sensor model

A real sensor is imperfect.

A simple model is:

$$
P_m(t)
=
P(t)
+
b
+
n(t)
$$

where:

- $b$ is bias;
- $n(t)$ is measurement noise.

In [ ]:
rng = np.random.default_rng(42)

sensor_bias_pa = 150.0
sensor_noise_std_pa = 250.0

P_true = pressure_from_depth(z_true)
P_meas = (
    P_true
    + sensor_bias_pa
    + rng.normal(0.0, sensor_noise_std_pa, size=len(t))
)

# 5. Convert measured pressure to estimated depth

In [ ]:
z_meas = depth_from_pressure(P_meas)

plt.plot(t/60, z_true, label="True depth")
plt.plot(t/60, z_meas, alpha=0.45, label="Depth from pressure sensor")

plt.gca().invert_yaxis()
plt.xlabel("Time [min]")
plt.ylabel("Depth [m]")
plt.title("True and measured depth")
plt.grid(True)
plt.legend()
plt.show()

The computer does not know the true curve.

It only sees the sensor-derived depth estimate.

# 6. Sampling

A dive computer works with discrete samples:

$$
z_k=z(t_k).
$$

The sampling interval might be:

$$
\Delta t=t_{k+1}-t_k.
$$

Sampling creates a discrete time-depth history:

$$
\{(t_k,z_k)\}.
$$

The profile history matters because many useful quantities depend not just on current depth, but on how the diver arrived there.

# 7. Estimate vertical velocity

From sampled depth:

$$
v
=
-\dot z.
$$

A simple estimate is:

$$
\hat v_k
\approx
-\frac{z_k-z_{k-1}}{\Delta t}.
$$

In [ ]:
dt = 1.0

v_true = -np.gradient(z_true, dt)
v_raw = -np.gradient(z_meas, dt)

plt.plot(t/60, v_true, label="True vertical velocity")
plt.plot(t/60, v_raw, alpha=0.45, label="Raw estimate from measured depth")

plt.xlabel("Time [min]")
plt.ylabel("Upward velocity [m/s]")
plt.title("Velocity estimated from depth history")
plt.grid(True)
plt.legend()
plt.show()

As we saw in Notebook 06:

> differentiation amplifies measurement noise.

So the computer may filter depth before estimating ascent or descent rate.

# 8. Low-pass filtering

In [ ]:
def low_pass(signal, alpha=0.15):
    y = np.zeros_like(signal)
    y[0] = signal[0]

    for k in range(1, len(signal)):
        y[k] = alpha*signal[k] + (1-alpha)*y[k-1]

    return y

z_filt = low_pass(z_meas, alpha=0.12)
v_filt = -np.gradient(z_filt, dt)

In [ ]:
plt.plot(t/60, v_true, label="True velocity")
plt.plot(t/60, v_raw, alpha=0.25, label="Raw estimate")
plt.plot(t/60, v_filt, label="Filtered estimate")

plt.xlabel("Time [min]")
plt.ylabel("Upward velocity [m/s]")
plt.title("Filtering improves velocity estimate")
plt.grid(True)
plt.legend()
plt.show()

Filtering reduces noise but introduces lag.

This is exactly the tradeoff studied in Notebooks 05–07:

$$
\boxed{
\text{noise reduction}
\leftrightarrow
\text{delay}
}
$$

# 9. What is measured and what is derived?

A useful distinction is:

### Directly measured

Pressure:

$$
P_m.
$$

### Derived from measurement

Depth:

$$
\hat z.
$$

Vertical velocity:

$$
\hat v.
$$

### Model-based

Gas forecast, tissue loading, decompression information, predicted limits.

The further we move down the pipeline, the more the result depends on assumptions and models.

# 10. Time-depth history as a state history

A current depth value alone does not summarize the dive.

Two divers can both be at 20 m now but have very different histories.

Therefore the computer stores or reconstructs a history:

$$
z(t_0),z(t_1),\ldots,z(t_k).
$$

Model outputs may depend on the entire past trajectory.

# 11. Cumulative depth-time exposure

As a simple example, define:

$$
E(t)
=
\int_0^t z(\tau)\,d\tau.
$$

This is not a decompression model.

It is only an example of a history-dependent quantity.

In [ ]:
depth_exposure = np.cumsum(z_filt)*dt

plt.plot(t/60, depth_exposure)
plt.xlabel("Time [min]")
plt.ylabel("Cumulative depth-time exposure [m·s]")
plt.title("A simple history-dependent integral")
plt.grid(True)
plt.show()

Many important diving models have this general structure:

$$
\text{history}
\rightarrow
\text{internal model state}
\rightarrow
\text{displayed quantity}.
$$

# 12. Gas integration

Notebook 19 modeled gas remaining as:

$$
\dot G=-q(z,t).
$$

A computer connected to tank-pressure information could combine:

- depth;
- pressure;
- recent gas-use rate;
- time.

Again, some values are measured and others estimated.

In [ ]:
def ambient_pressure_bar(depth_m):
    return 1 + depth_m/10

surface_rate_lpm = 18.0

gas_rate = surface_rate_lpm * ambient_pressure_bar(z_filt)

plt.plot(t/60, gas_rate)
plt.xlabel("Time [min]")
plt.ylabel("Estimated gas use [surface L/min]")
plt.title("Model-based gas-use estimate from depth")
plt.grid(True)
plt.show()

# 13. Integrate estimated gas consumption

If initial gas is:

$$
G_0,
$$

then:

$$
G(t)
=
G_0
-
\int_0^t q(\tau)d\tau.
$$

In [ ]:
G0 = 2400.0
dt_min = dt/60

G_est = G0 - np.cumsum(gas_rate)*dt_min
G_est = np.maximum(G_est, 0)

plt.plot(t/60, G_est)
plt.xlabel("Time [min]")
plt.ylabel("Estimated remaining gas [surface L]")
plt.title("Gas remaining from model integration")
plt.grid(True)
plt.show()

This estimate is only as good as:

- the breathing-rate assumption;
- the depth estimate;
- the gas model.

The display may look precise, but the underlying quantity is still model-dependent.

# 14. Current measurement vs future prediction

Suppose the computer estimates current gas-use rate:

$$
q_k.
$$

A simple forecast assumes current conditions continue:

$$
\hat G(t+\Delta t)
=
G(t)-q_k\Delta t.
$$

This is a prediction, not a measurement.

In [ ]:
forecast_horizon_min = 5.0

sample_idx = 700

current_depth = z_filt[sample_idx]
current_rate = gas_rate[sample_idx]
current_gas = G_est[sample_idx]

forecast_gas = current_gas - current_rate*forecast_horizon_min

print(f"Current depth: {current_depth:.2f} m")
print(f"Current estimated gas rate: {current_rate:.1f} L/min")
print(f"Current estimated gas: {current_gas:.0f} L")
print(f"5-min constant-condition forecast: {forecast_gas:.0f} L")

A dive computer is therefore constantly mixing three categories:

$$
\boxed{\text{measurement}}
$$

$$
\boxed{\text{state estimation}}
$$

$$
\boxed{\text{prediction}}
$$

# 15. Decompression information is model-based

A decompression model does not directly measure tissue gas loading.

Instead, it estimates hidden physiological states from the depth-time history.

Conceptually:

```text
pressure/depth history
        |
        v
decompression model
        |
        v
estimated tissue states
        |
        v
displayed guidance / limits
```

This is another state-estimation problem.

# 16. Hidden states

The computer may display information based on states that are not directly measurable.

For example, a simplified tissue model might contain:

$$
P_{t,1},P_{t,2},\ldots,P_{t,n}.
$$

These are internal model states.

The sensor does not measure them.

The model computes them.

This is strongly analogous to the observer and Kalman-filter ideas:

$$
\boxed{
\text{measure what you can}
+
\text{use a model}
\rightarrow
\text{estimate what you cannot measure}
}
$$

# 17. A toy hidden-state model

To illustrate the architecture, consider one hidden state:

$$
x_h.
$$

Let it follow:

$$
\dot x_h
=
\frac{z-x_h}{\tau_h}.
$$

This is not a physiological decompression model.

It is only a mathematical example of a hidden state driven by depth history.

In [ ]:
tau_h = 300.0  # s

x_hidden = np.zeros_like(t, dtype=float)

for k in range(len(t)-1):
    dx = (z_filt[k] - x_hidden[k])/tau_h
    x_hidden[k+1] = x_hidden[k] + dx*dt

plt.plot(t/60, z_filt, label="Depth input")
plt.plot(t/60, x_hidden, label="Hidden model state")

plt.xlabel("Time [min]")
plt.ylabel("State")
plt.title("A hidden state driven by dive history")
plt.grid(True)
plt.legend()
plt.show()

The hidden state responds gradually and remembers previous depth.

This is exactly the kind of dynamic structure we will formalize in Notebook 21 with decompression compartments.

# 18. Alarms as threshold logic

A computer may generate an alert when an estimated quantity crosses a threshold.

Conceptually:

$$
\hat v>v_{\text{limit}}
$$

or:

$$
G<G_{\text{threshold}}.
$$

This introduces discrete logic on top of continuous dynamics.

In [ ]:
v_limit_demo = 0.18

alarm = np.abs(v_filt) > v_limit_demo

plt.plot(t/60, alarm.astype(int))
plt.xlabel("Time [min]")
plt.ylabel("Alarm state")
plt.title("Example threshold-based alarm logic")
plt.grid(True)
plt.show()

This is another hybrid-system structure:

- continuous state estimation;
- discrete alerts and mode changes.

# 19. Sensor failure revisited

Notebook 06 introduced:

- bias;
- drift;
- dropout;
- frozen sensor.

A dive computer must decide whether sensor data remain plausible.

For example, if pressure is frozen while other signals suggest movement, the measurement may be inconsistent.

A fault-detection architecture might compare:

$$
\text{measured pressure}
$$

against:

$$
\text{model-predicted pressure}.
$$

The difference is a residual:

$$
r=P_m-\hat P.
$$

Persistent or implausible residuals can indicate a fault.

# 20. Architecture of a dive computer

Conceptually:

```text
PRESSURE SENSOR
      |
      v
DEPTH ESTIMATION
      |
      v
FILTERING + RATE ESTIMATION
      |
      v
TIME-DEPTH HISTORY
      |
      +------> GAS MODEL
      |
      +------> DECOMPRESSION MODEL
      |
      +------> ALARM LOGIC
      |
      v
DISPLAYED INFORMATION
```

Different blocks operate on different time scales and rely on different assumptions.

# 21. Measurement hierarchy

We can classify displayed values by how many modeling layers separate them from the sensor.

### Layer 1

Pressure.

### Layer 2

Depth.

### Layer 3

Vertical rate.

### Layer 4

Integrated or history-dependent quantities.

### Layer 5

Predictions and hidden-state model outputs.

The deeper the layer, the more model dependence enters.

# 22. Why sampling frequency matters

Suppose depth changes quickly.

A slow sampling rate may miss important dynamics.

A very fast sampling rate captures more detail but also exposes more sensor noise.

Again we see the tradeoff:

$$
\boxed{
\text{resolution}
\leftrightarrow
\text{noise}
}
$$

# 23. Compare sampling rates

In [ ]:
sample_intervals = [1, 5, 15]

for ds in sample_intervals:
    idx = np.arange(0, len(t), ds)

    plt.plot(
        t[idx]/60,
        z_meas[idx],
        marker="o",
        markersize=2,
        label=f"sample every {ds} s"
    )

plt.gca().invert_yaxis()
plt.xlabel("Time [min]")
plt.ylabel("Measured depth [m]")
plt.title("Effect of sampling interval")
plt.grid(True)
plt.legend()
plt.show()

# 24. The dive computer as an observer

A control-theory interpretation is:

> the dive computer is an observer of the dive state.

It receives partial, noisy measurements and reconstructs useful internal variables.

Some states are physical:

$$
z,\quad v.
$$

Some are resource states:

$$
G.
$$

Some are model states:

$$
P_{t,i}.
$$

This makes the architecture very close to:

$$
\boxed{
\text{sensor}
\rightarrow
\text{estimator}
\rightarrow
\text{model states}
\rightarrow
\text{decision information}
}
$$

# 25. What a dive computer does not know exactly

Even a sophisticated computer does not directly know:

- true future workload;
- future breathing demand;
- exact individual physiology;
- future movement;
- all model uncertainty.

Therefore displayed predictions should be understood as outputs of a model under assumptions.

# 26. Systems-engineering lesson

A precise digital display can create the impression of exact knowledge.

But system outputs can be divided into:

- measured;
- estimated;
- integrated;
- predicted.

Understanding which category a value belongs to is a core engineering skill.

# Exercises

### 1. Pressure-to-depth conversion

Generate pressure values corresponding to:

$$
0,\ 10,\ 20,\ 30\ \text{m}.
$$

Convert them back to depth.

Verify consistency.

In [ ]:
# Your code here

### 2. Sensor bias

Increase pressure-sensor bias.

How much depth error does it create?

In [ ]:
# Your code here

### 3. Velocity estimation

Compare raw differentiation with filtered differentiation for several filter strengths.

What is the tradeoff?

In [ ]:
# Your code here

### 4. Sampling rate

Estimate vertical velocity using:

- 1 s samples;
- 5 s samples;
- 15 s samples.

Which estimate responds fastest?
Which is noisiest?

In [ ]:
# Your code here

### 5. Hidden state

Change:

$$
\tau_h.
$$

How does the hidden-state memory change?

In [ ]:
# Your code here

# Challenge — build a mini dive-computer pipeline

Create a simulation that:

1. generates a true dive profile;
2. converts depth to noisy pressure;
3. reconstructs filtered depth;
4. estimates vertical velocity;
5. integrates gas use;
6. evolves one hidden model state;
7. raises a simple threshold alert;
8. displays all estimated quantities.

The key challenge is to keep separate:

$$
\boxed{\text{truth}}
$$

$$
\boxed{\text{measurement}}
$$

$$
\boxed{\text{estimate}}
$$

$$
\boxed{\text{prediction}}
$$

In [ ]:
# Your code here

# Summary

A dive computer can be understood as a measurement-and-model system.

Its basic architecture is:

$$
\boxed{
\text{pressure sensor}
\rightarrow
\text{depth estimate}
\rightarrow
\text{history}
\rightarrow
\text{models}
\rightarrow
\text{display}
}
$$

We learned that:

- pressure is measured;
- depth is derived;
- velocity is estimated from depth history;
- filtering reduces noise but introduces lag;
- gas remaining can be integrated from a consumption model;
- decompression states are hidden model states;
- alerts add discrete logic;
- sampling rate affects estimation quality;
- predictions are model-dependent.

### Core insight

$$
\boxed{
\text{measured}
\neq
\text{estimated}
\neq
\text{predicted}
}
$$

A dive computer combines all three.

### Next — Notebook 21

The next roadmap topic is **Decompression Models**.

We will introduce tissue compartments as first-order dynamical systems:

$$
\dot P_t
=
k(P_a-P_t)
$$

with:

$$
k=\frac{\ln 2}{T_{1/2}}.
$$

That will connect decompression directly to:

- differential equations;
- time constants;
- exponential responses;
- state-space models;
- numerical integration;
- hidden-state estimation.